# Intern KPI Insight Analysis Notebook

This notebook expands the KPI export into a more complete analysis package for HR.

It focuses on four outcomes:
- compare actual KPI scores against configured goals/targets
- identify which interns are on track or need attention
- generate charts that are suitable for PDF export
- produce descriptive insight tables that support HR review and coaching decisions


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.titleweight"] = "bold"

# Update this file to the KPI CSV exported from Odoo.
CSV_PATH = Path("intern_kpi_summary.csv")
OUTPUT_DIR = Path("exports")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
def pick_column(dataframe, candidates):
    for column in candidates:
        if column in dataframe.columns:
            return column
    raise KeyError(f"None of these columns were found: {candidates}")


employee_col = pick_column(df, ["Employee Name", "employee_name", "Intern", "name"])
avg_score_col = pick_column(df, ["Average Score", "avg_score"])
timeliness_score_col = pick_column(df, ["Timeliness Score", "timeliness_score"])
responsiveness_score_col = pick_column(df, ["Responsiveness Score", "responsiveness_score"])
punctuality_score_col = pick_column(df, ["Punctuality Score", "punctuality_score"])
quantity_score_col = pick_column(df, ["Quantity Score", "quantity_score"])
quality_score_col = pick_column(df, ["Quality Score", "quality_score"])
effectiveness_score_col = pick_column(df, ["Effectiveness Score", "effectiveness_score"])
efficiency_score_col = pick_column(df, ["Efficiency Score", "efficiency_score"])
accuracy_score_col = pick_column(df, ["Accuracy Score", "accuracy_score"])

metric_columns = {
    "Timeliness": timeliness_score_col,
    "Responsiveness": responsiveness_score_col,
    "Punctuality": punctuality_score_col,
    "Quantity": quantity_score_col,
    "Quality": quality_score_col,
    "Effectiveness": effectiveness_score_col,
    "Efficiency": efficiency_score_col,
    "Accuracy": accuracy_score_col,
}

target_percentage_columns = {
    metric: pick_column(df, [f"{metric}_target_percentage".lower(), f"{metric}_target_percentage", f"{metric} Target %", f"{metric} Target"])
    for metric in metric_columns
}

analysis_df = df[[employee_col, avg_score_col] + list(metric_columns.values()) + list(target_percentage_columns.values())].copy()
analysis_df = analysis_df.loc[:, ~analysis_df.columns.duplicated()].copy()
analysis_df = analysis_df.rename(columns={employee_col: "Employee", avg_score_col: "Average Score"})

analysis_df.head()

In [ ]:
def target_percentage_to_score(value):
    numeric_value = pd.to_numeric(value, errors="coerce")
    return ((numeric_value.fillna(0) / 100.0) * 5.0).round(2)


for metric, score_col in metric_columns.items():
    target_col = target_percentage_columns[metric]
    analysis_df[f"{metric} Score"] = pd.to_numeric(df[score_col], errors="coerce").fillna(0)
    analysis_df[f"{metric} Target %"] = pd.to_numeric(df[target_col], errors="coerce").fillna(0)
    analysis_df[f"{metric} Target Score"] = target_percentage_to_score(analysis_df[f"{metric} Target %"])
    analysis_df[f"{metric} Gap"] = (analysis_df[f"{metric} Target Score"] - analysis_df[f"{metric} Score"]).round(2)
    analysis_df[f"{metric} Status"] = analysis_df[f"{metric} Gap"].apply(lambda gap: "Meeting Target" if gap <= 0 else "Below Target")

gap_columns = [f"{metric} Gap" for metric in metric_columns]
analysis_df["Metrics Below Target"] = (analysis_df[gap_columns] > 0).sum(axis=1)
analysis_df["Overall Status"] = analysis_df["Metrics Below Target"].apply(lambda value: "On Track" if value == 0 else "Needs Attention")

analysis_df[["Employee", "Average Score", "Metrics Below Target", "Overall Status"]].sort_values(["Overall Status", "Average Score"], ascending=[True, False])

In [ ]:
def top_focus_areas(row):
    gaps = {metric: row[f"{metric} Gap"] for metric in metric_columns}
    ordered = sorted(gaps.items(), key=lambda item: item[1], reverse=True)
    return ", ".join(metric for metric, gap in ordered if gap > 0) or "All metrics meeting target"


insight_table = analysis_df[["Employee", "Average Score", "Overall Status", "Metrics Below Target"]].copy()
insight_table["Top Focus Areas"] = analysis_df.apply(top_focus_areas, axis=1)
insight_table["HR Insight"] = analysis_df.apply(
    lambda row: (
        f"{row['Employee']} is currently on track across all configured KPIs. Continue reinforcing current work habits and monitor consistency."
        if row["Overall Status"] == "On Track"
        else f"{row['Employee']} needs coaching in {top_focus_areas(row)}. The current KPI profile suggests targeted follow-up and short review cycles are needed."
    ),
    axis=1,
)

insight_table.sort_values(["Overall Status", "Average Score"], ascending=[True, False])

## Chart 1: Average KPI Scores vs Target Scores

This chart helps HR compare the team's average actual KPI score against the average configured score threshold.

In [ ]:
metric_summary = pd.DataFrame(
    {
        "Metric": list(metric_columns.keys()),
        "Average Actual Score": [analysis_df[f"{metric} Score"].mean() for metric in metric_columns],
        "Average Target Score": [analysis_df[f"{metric} Target Score"].mean() for metric in metric_columns],
    }
)

plot_df = metric_summary.melt(id_vars="Metric", var_name="Type", value_name="Score")

fig, ax = plt.subplots(figsize=(13, 7))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="Type", ax=ax)
ax.set_ylim(0, 5.2)
ax.set_title("Average KPI Scores vs Target Scores")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.xticks(rotation=20)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "kpi_scores_vs_targets.png", dpi=220, bbox_inches="tight")
plt.show()

## Chart 2: Timeliness vs Responsiveness Scatterplot

A two-dimensional view that makes it easier to identify interns who are both deadline-conscious and communication-responsive.

In [ ]:
scatter_df = analysis_df[["Employee", "Timeliness Score", "Responsiveness Score", "Overall Status"]].copy()

fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(
    data=scatter_df,
    x="Timeliness Score",
    y="Responsiveness Score",
    hue="Overall Status",
    s=180,
    ax=ax,
)

for _, row in scatter_df.iterrows():
    ax.annotate(row["Employee"], (row["Timeliness Score"], row["Responsiveness Score"]), textcoords="offset points", xytext=(6, 6), fontsize=9)

ax.set_title("Timeliness vs Responsiveness")
ax.set_xlim(0, 5.2)
ax.set_ylim(0, 5.2)
ax.axvline(scatter_df["Timeliness Score"].mean(), linestyle="--", color="#64748b", linewidth=1)
ax.axhline(scatter_df["Responsiveness Score"].mean(), linestyle="--", color="#64748b", linewidth=1)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "timeliness_vs_responsiveness.png", dpi=220, bbox_inches="tight")
plt.show()

## Chart 3: KPI Gap Heatmap

Positive values indicate how far an intern is below target. Zero or negative values mean the target is being met or exceeded.

In [ ]:
heatmap_df = analysis_df[["Employee"] + [f"{metric} Gap" for metric in metric_columns]].copy()
heatmap_df = heatmap_df.set_index("Employee")
heatmap_display = heatmap_df.clip(lower=0)

fig, ax = plt.subplots(figsize=(13, max(5, len(heatmap_display) * 0.8)))
sns.heatmap(heatmap_display, annot=True, cmap="OrRd", linewidths=0.5, cbar_kws={"label": "Target Gap"}, ax=ax)
ax.set_title("KPI Gap Heatmap")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "kpi_gap_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

## Chart 4: Overall Status Distribution

A quick management view of how many interns are fully meeting KPI targets versus those still needing intervention.

In [ ]:
status_counts = analysis_df["Overall Status"].value_counts().rename_axis("Status").reset_index(name="Count")

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=status_counts, x="Status", y="Count", ax=ax)
ax.set_title("Intern Performance Status Distribution")
ax.set_xlabel("")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "performance_status_distribution.png", dpi=220, bbox_inches="tight")
plt.show()

## Export Tables for PDF or Management Review

In [ ]:
metric_status_columns = [
    "Employee",
    "Average Score",
    "Overall Status",
]

for metric in metric_columns:
    metric_status_columns.extend([
        f"{metric} Score",
        f"{metric} Target Score",
        f"{metric} Gap",
        f"{metric} Status",
    ])

detailed_export = analysis_df[metric_status_columns].copy()
detailed_export.to_csv(OUTPUT_DIR / "intern_kpi_detailed_analysis.csv", index=False)
insight_table.to_csv(OUTPUT_DIR / "intern_kpi_hr_insights.csv", index=False)

display(detailed_export.head())
display(insight_table.head())
OUTPUT_DIR